In [1]:
import pandas as pd

In [2]:
import numpy as np
import lightgbm as lgm

In [3]:
data = pd.read_csv(r"D:\Documents\DEPI\DATA\data\train_before_2016.csv")

C:\Users\Reem\AppData\Local\Temp\ipykernel_21844\1063481803.py:1: DtypeWarning: Columns (0: event_name_2, 1: event_type_2) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(r"D:\Documents\DEPI\DATA\data\train_before_2016.csv")


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 54821020 entries, 0 to 54821019
Data columns (total 21 columns):
 #   Column        Dtype
---  ------        -----
 0   item_id       str  
 1   store_id      str  
 2   dept_id       str  
 3   cat_id        str  
 4   state_id      str  
 5   day           str  
 6   sales         int64
 7   date          str  
 8   wm_yr_wk      int64
 9   weekday       str  
 10  wday          int64
 11  month         int64
 12  year          int64
 13  d             str  
 14  event_name_1  str  
 15  event_type_1  str  
 16  event_name_2  str  
 17  event_type_2  str  
 18  snap_CA       int64
 19  snap_TX       int64
 20  snap_WI       int64
dtypes: int64(8), str(13)
memory usage: 8.6 GB


In [5]:
data.head()

,item_id,store_id,dept_id,cat_id,state_id,day,sales,date,wm_yr_wk,weekday,...,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,d_1,3,2011-01-29,11101,Saturday,...,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,d_2,0,2011-01-30,11101,Sunday,...,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,d_3,0,2011-01-31,11101,Monday,...,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,d_4,1,2011-02-01,11101,Tuesday,...,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,d_5,4,2011-02-02,11101,Wednesday,...,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [6]:
data.drop(columns=["Unnamed: 0"], inplace=True, errors="ignore")

# Convert date
data["date"] = pd.to_datetime(data["date"])

data = data.sort_values(["item_id", "store_id", "date"])

# Time features
data["dayofweek"] = data["date"].dt.dayofweek
data["week"] = data["date"].dt.isocalendar().week.astype(int)

# Lag features
data["lag_1"] = data.groupby(["item_id", "store_id"])["sales"].shift(1)
data["lag_7"] = data.groupby(["item_id", "store_id"])["sales"].shift(7)

# Rolling mean
data["rolling_mean_7"] = (
    data.groupby(["item_id", "store_id"])["sales"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

data[["lag_1", "lag_7", "rolling_mean_7"]] = \
    data[["lag_1", "lag_7", "rolling_mean_7"]].fillna(0)

# Categorical Features
cat_cols = [
    "item_id", "store_id", "dept_id", "cat_id", "state_id",
    "event_name_1", "event_type_1",
    "event_name_2", "event_type_2",
    "weekday"
]

# Fill missing categorical values
data[cat_cols] = data[cat_cols].fillna("None")

# Convert to category dtype
for col in cat_cols:
    data[col] = data[col].astype("category")

data = data.drop(columns=["date"])
data = data.drop(columns=["weekday"])

In [7]:
data.head()

,item_id,store_id,dept_id,cat_id,state_id,day,sales,wm_yr_wk,wday,month,...,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,dayofweek,week,lag_1,lag_7,rolling_mean_7
0,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,d_1,3,11101,1,1,...,None,None,0,0,0,5,4,0.0,0.0,0.0
1,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,d_2,0,11101,2,1,...,None,None,0,0,0,6,4,3.0,0.0,0.0
2,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,d_3,0,11101,3,1,...,None,None,0,0,0,0,5,0.0,0.0,0.0
3,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,d_4,1,11101,4,2,...,None,None,1,1,0,1,5,0.0,0.0,0.0
4,FOODS_1_001,CA_1,FOODS_1,FOODS,CA,d_5,4,11101,5,2,...,None,None,1,0,1,2,5,1.0,0.0,0.0


In [8]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(data, test_size=0.2, random_state=42, shuffle=True)

In [9]:
x_train = train.drop("sales", axis=1)
y_train = train["sales"]

x_test = test.drop("sales", axis=1)
y_test = test["sales"]

In [10]:
x_train = x_train.replace([np.inf, -np.inf], 0)
y_train = y_train.astype(float)

x_test = x_test.replace([np.inf, -np.inf], 0)
y_test = y_test.astype(float)

In [12]:
x_train['d'] = x_train['d'].str.replace('d_', '').astype(int)
x_test['d'] = x_test['d'].str.replace('d_', '').astype(int)

In [14]:
x_train['day'] = x_train['day'].str.replace('d_', '').astype(int)
x_test['day'] = x_test['day'].str.replace('d_', '').astype(int)

In [15]:
model = lgm.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6
)

model.fit(x_train, y_train)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 3.002714 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4353
[LightGBM] [Info] Number of data points in the train set: 43856816, number of used features: 23
[LightGBM] [Info] Start training from score 1.111661


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,6
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [16]:
y_pred = model.predict(x_test)

In [17]:
import sklearn.metrics

mae = sklearn.metrics.mean_absolute_error(y_test, y_pred)
print("MAE:", mae)
mse = sklearn.metrics.mean_squared_error(y_test, y_pred)
print("MSE:", mse)

MAE: 0.7843746056224916
MSE: 4.113793940578163
